# Chapter 4 (part 4) — Regression Diagnostics: How Much Should I Trust a Stock's Beta?

Covers: Regression Diagnostics → Outliers, Influential Values, Heteroskedasticity, Non-Normality and Correlated Errors, Partial Residual Plots and Nonlinearity, on the **markets** data.

**Chapter thread, continued:** notebooks 08-10 asked what a coefficient means; here we ask whether it deserves trust. **Project question: how much should I trust a stock's beta?** Beta is the slope of a stock's daily excess return (return minus the risk-free rate) on the market's excess return (`Mkt_RF`, from Fama-French). The data: about 2,500 trading days (2016-2025) for 66 large-cap US stocks plus the three Fama-French factors. It is a time series with fat tails and turbulent stretches, which is exactly where textbook assumptions strain.

**How to work each section** (~10-15 min each): write your one-line prediction **before** running anything, do the code, then write 2-3 bullets on what matched and what didn't. Ask for a hint if you get stuck. The exam for this notebook lives in `reviews/R01_regression-diagnostics.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from statsmodels.nonparametric.smoothers_lowess import lowess

pd.set_option("display.max_columns", None)

from mlstats.data import load_returns, load_factors

excess = load_returns(excess=True)   # daily simple return minus RF, one column per ticker
factors = load_factors()             # Mkt_RF, SMB, HML, RF (decimals)


def stock_frame(ticker):
    """One stock's daily excess return as `y`, next to the three factors."""
    return pd.concat([excess[ticker].rename("y"), factors[["Mkt_RF", "SMB", "HML"]]], axis=1)


low = stock_frame("LOW")   # sections 1-3
spg = stock_frame("SPG")   # section 4
low.shape, low.index.min().date(), low.index.max().date()

## 1. Outliers and Influential Values

Covers: *Outliers* and *Influential Values* (book: Regression Diagnostics). A big residual and a big influence are different things. Numeric picture: the fitted line is a seesaw balanced on the cloud of points. About 2,500 days sit within a couple of percent of a market move of zero; moving one of them by 5 points barely tilts the line. One day with the market at −6% sits far out on the plank, so moving *it* by 5 points tilts the whole line. Influence is roughly **leverage** (how far out on the plank, a function of x only) × **how surprising the residual is**. Cook's distance combines the two; the studentized residual measures only the surprise.

**Predict first:** if you drop the single day with the largest Cook's distance, will LOW's beta move by less than 0.01, by about 0.05, or by more than 0.15? And will that day be a company earnings date or a market-wide crash day?

**Question 1.** (a) Fit LOW's excess return on `Mkt_RF`. (b) Build a DataFrame `diag` (indexed by date) with Cook's distance, leverage, and the externally studentized residual for every day; the cell after it prints the top days and plots them. (c) Refit dropping the 1, 3, and 10 most influential days (by Cook's distance), and also dropping every day from 2020-02-15 to 2020-04-30. Collect the betas in `betas`.

*Tools for this question:*
- `smf.ols("y ~ Mkt_RF", df).fit()`; `res.get_influence()` returns an `OLSInfluence` object.
- On it: `.cooks_distance` is a **tuple** `(distances, p_values)`, take `[0]`; `.hat_matrix_diag` is leverage; `.resid_studentized_external` is the studentized residual. All three are plain numpy arrays, so pass `index=df.index` when you build the DataFrame.
- `diag["cooks"].nlargest(k).index` gives the k dates; `df.drop(index=dates)` removes them.

_Prediction (one line, before running anything):_

In [ ]:
model = ...  # OLS of LOW's excess return y on Mkt_RF

In [ ]:
infl = ...  # the influence object from the fitted model
diag = ...  # DataFrame indexed by date with columns: cooks, leverage, stud_resid

In [ ]:
# pre-filled: consumes your `diag`
top = diag.join(low[["y", "Mkt_RF"]]).sort_values("cooks", ascending=False).head(8)
print(top.round(4))
print("rule-of-thumb cutoff 4/n =", round(4 / len(low), 4), "| days above it:", int((diag["cooks"] > 4 / len(low)).sum()), "of", len(low))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(diag["leverage"], diag["stud_resid"], s=10 + 400 * diag["cooks"] / diag["cooks"].max(), alpha=0.4)
for d in top.index[:3]:
    ax.annotate(str(d.date()), (diag.loc[d, "leverage"], diag.loc[d, "stud_resid"]), fontsize=8)
ax.set_xlabel("leverage"); ax.set_ylabel("studentized residual"); ax.set_title("LOW: bubble size = Cook's distance")
plt.show()

In [ ]:
betas = {}  # label -> LOW beta, e.g. betas['all days'], betas['drop top 1'], ..., betas['drop Feb15-Apr30 2020']

In [ ]:
pd.Series(betas).round(4)  # pre-filled

_What matched, what surprised you, and why (2-3 bullets):_

## 2. Heteroskedasticity

Covers: *Heteroskedasticity, Non-Normality and Correlated Errors* (book). Classical OLS standard errors assume every day's residual has the same spread. Numeric picture of what breaks: the slope's uncertainty is driven by `x² · e²` summed over days. A typical day (market move 1%, residual 1%) contributes 1e-4 × 1e-4 = 1e-8. A wild day (market −6%, residual −5%) contributes 0.0036 × 0.0025 = 9e-6, about 900 typical days' worth. If big residuals live on big-|x| days, the classical SE (one averaged σ² for everybody) is wrong, while **HC3** robust SEs (which use each day's own residual and leverage) are not.

Careful what Breusch-Pagan tests: as called here it checks whether the residual *variance moves linearly with the regressor* `Mkt_RF`. It is blind to variance that rises on big days in *both* directions (add `Mkt_RF**2` to its auxiliary regression and it becomes White's test), and it says nothing about variance drifting over *time*. HC3 stays sensitive to the `x² · e²` pattern either way. Clustering in time is section 3.

**Predict first:** will HC3's SE for beta be about the same as the classical one, about 1.5× bigger, or more than 2× bigger? Does the beta estimate itself change? And if you remove the single most influential day from section 1, will Breusch-Pagan still reject?

**Question 2.** (a) Run Breusch-Pagan on your section-1 `model`. (b) Refit with HC3 standard errors as `model_hc3`; the pre-filled table compares them. (c) Repeat (a) and (b) on `low_wo_top` (LOW without its most influential day, built for you), storing `bp_wo` and `model_hc3_wo`.

*Tools for this question:*
- `het_breuschpagan(resid, exog_het, robust=True)` returns **4 values** `(lm_stat, lm_pvalue, f_stat, f_pvalue)`. `exog_het` must include the constant column: use `model.model.exog`. The default `robust=True` is the Koenker version, which suits fat tails.
- HC3: `smf.ols(...).fit(cov_type="HC3")`; compare `.bse["Mkt_RF"]`. The coefficients are identical by construction.
- `het_white(resid, exog)` also returns 4 values `(lm_stat, lm_pvalue, f_stat, f_pvalue)`; the last pre-filled cell calls it for you. In your reflection, compare the BP p-value, the SE ratio, and the White p-value, with and without the top day.

_Prediction (one line, before running anything):_

In [ ]:
# pre-filled: residuals over time (consumes your `model`)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(model.resid.index, model.resid.values, lw=0.5)
ax.set_title("LOW: OLS residuals over time"); plt.show()

In [ ]:
bp = ...  # Breusch-Pagan on `model` (the 4-tuple)

In [ ]:
model_hc3 = ...  # same regression, HC3 standard errors

In [ ]:
# pre-filled: consumes `model` and `model_hc3`
print(pd.DataFrame({"coef": model.params, "classical SE": model.bse, "HC3 SE": model_hc3.bse,
                    "SE ratio": model_hc3.bse / model.bse}).round(4))
print("Breusch-Pagan p-value:", bp[1])

In [ ]:
low_wo_top = low.drop(index=diag["cooks"].idxmax())   # pre-filled
bp_wo = ...         # Breusch-Pagan on the regression without that day
model_hc3_wo = ...  # HC3 fit without that day

In [ ]:
# pre-filled
print("BP p-value without the top day:", bp_wo[1])
m_wo = smf.ols("y ~ Mkt_RF", low_wo_top).fit()
print("SE ratio (HC3 / classical) without the top day:", round(model_hc3_wo.bse["Mkt_RF"] / m_wo.bse["Mkt_RF"], 2))
print("White test p-value (with Mkt_RF squared in the auxiliary regression): all days",
      het_white(model.resid, model.model.exog)[1], "| without the top day", het_white(m_wo.resid, m_wo.model.exog)[1])

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Correlated Errors

Covers: *Correlated Errors* (book; it matters far more here than the book lets on). Two different failures live here.

**(i) Uncorrelated is not independent.** Numeric picture: residuals of +1%, −1%, +1%, −1% on calm days and +5%, −6%, +4%, −5% on turbulent days. Their *signs* are random, so the average product of neighbors is near zero and the residuals look uncorrelated. Yet the *sizes* clearly cluster (calm next to calm, wild next to wild), so the *squares* are correlated.

**(ii) Overlapping returns.** The 20-day return ending today and the one ending yesterday share 19 of their 20 daily returns, so their regression errors are almost the same number. Rule of thumb: Durbin-Watson ≈ 2(1 − ρ), where ρ is the correlation between neighboring residuals. With ρ = 0.5 that is DW ≈ 1; with ρ = 0.9, DW ≈ 0.2. The point estimates are still fine, but the *standard errors* are far too small. Newey-West (HAC) standard errors repair this, with `maxlags` set to the window length minus 1.

**Predict first:** the lag-1 autocorrelation of LOW's daily residuals: near 0, near +0.3, or near −0.3? Is the lag-1 autocorrelation of the *squared* residuals bigger or smaller in size? And will the overlapping 20-day regression's Durbin-Watson be near 2, near 1, or near 0.2?

**Question 3.** (a) Compute the autocorrelation of `model.resid` and of `model.resid ** 2` at lags 1-5 (`acf_e`, `acf_e2`). (b) Fit the overlapping 20-day regression on the pre-built frame `w` as `model20` and compute its Durbin-Watson as `dw20`. (c) Refit `model20` with HAC standard errors (`maxlags=19`) as `model20_hac`. The pre-filled cell compares it with non-overlapping windows, started at each of the 20 possible offsets. Notice that the 20-day beta is not the daily beta; say why in your reflection.

*Tools for this question:*
- `pd.Series.autocorr(lag=k)` for each k. Return a list of **5 values for lags 1-5** (no lag 0), since the pre-filled table assumes it. (`acorr_ljungbox(x, lags=[10])` returns a DataFrame with a joint p-value if you want a formal test.)
- `durbin_watson(res.resid)` takes the residual series.
- HAC: `.fit(cov_type="HAC", cov_kwds={"maxlags": 19})`.

_Prediction (one line, before running anything):_

In [ ]:
acf_e = ...   # lags 1-5 autocorrelations of the daily residuals
acf_e2 = ...  # lags 1-5 autocorrelations of the squared daily residuals
pd.DataFrame({"resid": np.asarray(acf_e), "resid^2": np.asarray(acf_e2)}, index=range(1, 6)).round(3)

In [ ]:
# pre-filled: 20-day windows built from a rolling SUM of daily excess returns
# (an approximation to compounding that is fine at this scale). Consecutive rows overlap by 19 days.
w = low[["y", "Mkt_RF"]].rolling(20).sum().dropna()
w.shape

In [ ]:
model20 = ...  # OLS of the 20-day y on the 20-day Mkt_RF
dw20 = ...     # Durbin-Watson of its residuals

In [ ]:
model20_hac = ...  # same regression with HAC standard errors, maxlags=19

In [ ]:
# pre-filled: consumes model20, dw20, model20_hac
print("Durbin-Watson, 20-day overlapping:", round(dw20, 3))
nonover = smf.ols("y ~ Mkt_RF", w.iloc[::20]).fit()   # every 20th row: no shared days
print(pd.DataFrame({"beta": [model20.params["Mkt_RF"], model20.params["Mkt_RF"], nonover.params["Mkt_RF"]],
                    "SE": [model20.bse["Mkt_RF"], model20_hac.bse["Mkt_RF"], nonover.bse["Mkt_RF"]],
                    "n": [len(w), len(w), len(w.iloc[::20])]},
                   index=["overlapping, classical SE", "overlapping, HAC SE", "non-overlapping (offset 0)"]).round(4))

# the same non-overlapping regression from each of the 20 possible starting offsets
offsets = pd.Series({k: smf.ols("y ~ Mkt_RF", w.iloc[k::20]).fit().params["Mkt_RF"] for k in range(20)})
print("beta across the 20 offsets: mean", round(offsets.mean(), 3), "| std", round(offsets.std(), 3),
      "| min/max", round(offsets.min(), 2), round(offsets.max(), 2))
print("daily beta from section 1:", round(model.params["Mkt_RF"], 3))

_What matched, what surprised you, and why (2-3 bullets):_

## 4. Partial Residual Plot and Nonlinearity

Covers: *Partial Residual Plots and Nonlinearity* (book). A partial residual isolates one predictor's contribution after the others have done their work: `partial residual for x = residual + b_x · x`. Numeric picture: if b_Mkt = 1.0, a day with Mkt_RF = −4% and a residual of +1% has partial residual −4% + 1% = −3%. Plot that against Mkt_RF for every day: if the true relationship is a straight line of slope b_Mkt, the cloud follows the line; if it bends, a smooth curve through the cloud (lowess) leaves the line.

The model is the Fama-French three-factor regression for SPG (Simon Property): excess return on `Mkt_RF`, `SMB`, `HML`. The question is whether its market slope is the same on up days and down days: "does beta go up when the market falls?" This sets up notebook 12 (splines).

**Predict first:** will SPG's market slope on down days be steeper, flatter, or about equal to its up-day slope? Will the "extra down-market slope" coefficient come out positive or negative? Will it survive if you refit without Feb 15 - Apr 30, 2020?

**Question 4.** (a) Fit `model3` (the three-factor regression on `spg`) and compute the partial residuals for `Mkt_RF` by hand as `pr`; the plot is pre-filled. (b) Add a down-market term, `min(Mkt_RF, 0)`, to the formula and refit as `model_hinge`. (c) Refit the same hinge model on `spg_calm` (the Covid window removed, built for you) as `model_hinge_calm`.

*Tools for this question:*
- The hinge column: `I(np.minimum(Mkt_RF, 0))` inside the formula. The slope on up days is `b_Mkt`; the slope on **down days is `b_Mkt + b_hinge`**, so the hinge coefficient is the *extra* down-day slope.
- `pr` should be a Series aligned with `spg.index`: use `model3.resid` and `model3.params["Mkt_RF"]`.
- Library check, once you have computed `pr` by hand: `sm.graphics.plot_ccpr(model3, "Mkt_RF")` draws the same plot.

_Prediction (one line, before running anything):_

In [ ]:
model3 = ...  # OLS of SPG excess return on Mkt_RF + SMB + HML
pr = ...      # partial residuals for Mkt_RF, by hand

In [ ]:
# pre-filled: consumes `pr` and `model3`
smooth = lowess(pr, spg["Mkt_RF"], frac=0.4)
xs = np.linspace(spg["Mkt_RF"].min(), spg["Mkt_RF"].max(), 50)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(spg["Mkt_RF"], pr, s=6, alpha=0.3)
ax.plot(xs, model3.params["Mkt_RF"] * xs, "r--", label="straight line (OLS slope)")
ax.plot(smooth[:, 0], smooth[:, 1], "k-", label="lowess")
ax.set_xlabel("Mkt_RF"); ax.set_ylabel("partial residual"); ax.legend(); ax.set_title("SPG: partial residual plot for Mkt_RF")
plt.show()

In [ ]:
model_hinge = ...  # three-factor model plus the down-market term

In [ ]:
spg_calm = spg[(spg.index < "2020-02-15") | (spg.index > "2020-04-30")]   # pre-filled
model_hinge_calm = ...  # the same hinge model on the calm sample

In [ ]:
# pre-filled: consumes both hinge models
hinge = lambda m: [p for p in m.params.index if "minimum" in p][0]
print(pd.DataFrame({"extra down-day slope": [model_hinge.params[hinge(model_hinge)], model_hinge_calm.params[hinge(model_hinge_calm)]],
                    "t": [model_hinge.tvalues[hinge(model_hinge)], model_hinge_calm.tvalues[hinge(model_hinge_calm)]],
                    "up-day slope (Mkt_RF)": [model_hinge.params["Mkt_RF"], model_hinge_calm.params["Mkt_RF"]]},
                   index=["all days", "Covid window removed"]).round(3))

_What matched, what surprised you, and why (2-3 bullets):_

**Where this leaves the project question:** write one sentence on how much you would trust LOW's daily beta, and which of the four problems (influence, heteroskedasticity, dependence, nonlinearity) mattered most for it.